# Model Switching & Cancel

This notebook shows how to:
- Route requests to different workers based on `model_id`
- Switch models mid-training using `discard()`
- Cancel individual requests with `cancel()`
- Handle worker failures gracefully

In [1]:
import time
import torch

from softlabels import (
    Broker, Worker, Client,
    BatchConfig, TensorSpec, EndpointConfig,
)

## Setup

One worker serves 3 model_ids. Each model produces different data
so we can verify routing is correct.

In [ ]:
config = BatchConfig([TensorSpec("x", (4,), "float32")])
endpoints = EndpointConfig()

broker = Broker(endpoints=endpoints)
broker.start()
time.sleep(0.2)

# Worker produces model_id-dependent data
MODEL_VALUES = {"layer_0": 0.0, "layer_1": 1.0, "layer_2": 2.0}

def generator(model_id: str) -> bytes:
    val = MODEL_VALUES[model_id]
    return config.encode(x=torch.full((4,), val))

worker = Worker(
    generator_fn=generator,
    model_ids=list(MODEL_VALUES.keys()),
    slot_size=config.nbytes(),
    endpoints=endpoints,
)
worker.start()
time.sleep(0.2)

client = Client(slot_count=8, batch_config=config, endpoints=endpoints)
client.hello()
print("Broker started, worker ready.")

## Model-aware routing

Requests are routed to the correct model. Each model returns its own value.

In [3]:
for model_id, expected_val in MODEL_VALUES.items():
    slot = client.request_sample(model_id, timeout_ms=2000)
    data = config.decode(client.medium.read(slot))
    client.release_slot(slot)
    assert data["x"][0].item() == expected_val
    print(f"{model_id}: {data['x']} ✓")

layer_0: tensor([0., 0., 0., 0.]) ✓
layer_1: tensor([1., 1., 1., 1.]) ✓
layer_2: tensor([2., 2., 2., 2.]) ✓


## Model switching with `discard()`

Simulate layer-by-layer training: train on `layer_0`, then switch to `layer_1`.
`discard()` cancels all pending requests and drains stale completions.

In [4]:
for layer_idx in range(3):
    model_id = f"layer_{layer_idx}"
    print(f"\nTraining on {model_id}:")
    
    for step in range(3):
        slot = client.request_sample(model_id, timeout_ms=2000)
        data = config.decode(client.medium.read(slot))
        client.release_slot(slot)
        assert data["x"][0].item() == float(layer_idx)
        print(f"  step {step}: x[0]={data['x'][0].item()} ✓")
    
    # Switch to next layer: discard old work
    cancelled = client.discard()
    print(f"  Discarded {cancelled} pending requests")


Training on layer_0:
  step 0: x[0]=0.0 ✓
  step 1: x[0]=0.0 ✓
  step 2: x[0]=0.0 ✓
  Discarded 0 pending requests

Training on layer_1:
  step 0: x[0]=1.0 ✓
  step 1: x[0]=1.0 ✓
  step 2: x[0]=1.0 ✓
  Discarded 0 pending requests

Training on layer_2:
  step 0: x[0]=2.0 ✓
  step 1: x[0]=2.0 ✓
  step 2: x[0]=2.0 ✓
  Discarded 0 pending requests


## Cancel a specific request

In [5]:
token = client.request_slot("layer_0")
print(f"Requested token: {token[:15]}...")
time.sleep(0.1)

ok = client.cancel(token)
print(f"Cancelled: {ok}")
print(f"Pending slots after cancel: {len(client._pending_slots)}")

Requested token: client-...-1
Cancelled: True
Pending slots after cancel: 0


## Worker replacement

Stop the current worker, start a new one. The broker detects the GOODBYE
immediately and routes to the replacement.

In [ ]:
worker.stop()
time.sleep(0.2)
print(f"Workers after stop: {broker.get_stats().connected_workers}")

# Start replacement
worker2 = Worker(
    generator_fn=generator,
    model_ids=list(MODEL_VALUES.keys()),
    slot_size=config.nbytes(),
    endpoints=endpoints,
)
worker2.start()
time.sleep(0.2)
print(f"Workers after new start: {broker.get_stats().connected_workers}")

slot = client.request_sample("layer_0", timeout_ms=2000)
data = config.decode(client.medium.read(slot))
client.release_slot(slot)
print(f"New worker serving: {data['x']} ✓")

## Graceful shutdown

In [7]:
client.close()
worker2.stop()
time.sleep(0.2)
broker.stop()
print("Shutdown complete.")

Shutdown complete.
